In [ ]:
# INSTALACIÓN
!pip install pyamg pandas matplotlib -q

# LIBRERÍAS

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import gmres
import pyamg
import pandas as pd
import time

# CONTADOR DE ITERACIONES

class IterCounter:
    def __init__(self):
        self.niter = 0

    def __call__(self, rk=None):
        self.niter += 1

# FUNCIÓN DE EXPERIMENTO

def ejecutar_metodo(A, b, M=None):

    contador = IterCounter()

    inicio = time.time()

    x, info = gmres(
        A,
        b,
        M=M,
        callback=contador,
        rtol=1e-6,
        maxiter=5000
    )

    tiempo = time.time() - inicio

    residual = np.linalg.norm(b - A @ x) / np.linalg.norm(b)

    return tiempo, contador.niter, residual

# TAMAÑOS DE MATRIZ

tamanos = [1024, 4096, 16384]

resultados = []

# EXPERIMENTOS

for n in tamanos:

    print(f"\nProcesando matriz n={n}")

    diag = 2*np.ones(n)
    off = -1*np.ones(n-1)

    A = sp.diags(
        [off, diag, off],
        [-1, 0, 1],
        format='csr'
    )

    x_real = np.ones(n)

    b = A @ x_real

    # GMRES

    tiempo, iters, residual = ejecutar_metodo(A, b)

    resultados.append([
        n,
        "GMRES",
        tiempo,
        iters,
        residual
    ])

    # GMRES + AMG

    ml = pyamg.smoothed_aggregation_solver(A)

    M_amg = ml.aspreconditioner()

    tiempo, iters, residual = ejecutar_metodo(
        A,
        b,
        M=M_amg
    )

    resultados.append([
        n,
        "GMRES + AMG",
        tiempo,
        iters,
        residual
    ])

# TABLA FINAL

tabla = pd.DataFrame(
    resultados,
    columns=[
        "Tamaño",
        "Método",
        "Tiempo (s)",
        "Iteraciones",
        "Residual"
    ]
)

print("\nRESULTADOS FINALES\n")

display(tabla)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.5 MB/s eta 0:00:00

Procesando matriz n=1024


/tmp/ipykernel_3296/1964554255.py:72: DeprecationWarning: scipy.sparse.linalg.gmres called without specifying `callback_type`. The default value will be changed in a future release. For compatibility, specify a value for `callback_type` explicitly, e.g., ``gmres(..., callback_type='pr_norm')``, or to retain the old behavior ``gmres(..., callback_type='legacy')``
  tiempo, iters, residual = ejecutar_metodo(A, b)
/tmp/ipykernel_3296/1964554255.py:88: DeprecationWarning: scipy.sparse.linalg.gmres called without specifying `callback_type`. The default value will be changed in a future release. For compatibility, specify a value for `callback_type` explicitly, e.g., ``gmres(..., callback_type='pr_norm')``, or to retain the old behavior ``gmres(..., callback_type='legacy')``
  tiempo, iters, residual = ejecutar_metodo(



Procesando matriz n=4096

Procesando matriz n=16384

RESULTADOS FINALES



,Tamaño,Método,Tiempo (s),Iteraciones,Residual
0,1024,GMRES,1.699371,5000,1.204397e-04
1,1024,GMRES + AMG,0.011833,5,3.077026e-07
2,4096,GMRES,1.307832,5000,9.492424e-05
3,4096,GMRES + AMG,0.013466,7,8.761681e-08
4,16384,GMRES,3.840401,5000,9.492424e-05
5,16384,GMRES + AMG,0.046311,6,5.550909e-08


Los resultados obtenidos evidencian que el método GMRES sin precondicionamiento presenta dificultades para converger dentro del número máximo de iteraciones establecido, especialmente a medida que aumenta el tamaño de la matriz. Por otro lado, la incorporación del precondicionador AMG reduce significativamente el número de iteraciones requeridas y mejora la precisión de la solución, alcanzando residuales varios órdenes de magnitud menores. En consecuencia, el uso de AMG como precondicionador permite resolver sistemas lineales dispersos de gran tamaño de manera más eficiente y robusta que GMRES aplicado de forma directa.

In [ ]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 1024 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

python3: can't open file '/content/examples/all_in_one.py': [Errno 2] No such file or directory


In [ ]:
!git clone https://github.com/jiechenjiechen/GNP.git

Cloning into 'GNP'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 45 (delta 3), reused 1 (delta 1), pack-reused 32 (from 1)
Receiving objects: 100% (45/45), 13.19 MiB | 15.41 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [ ]:
%cd GNP

/content/GNP


In [ ]:
!pip install -e .

Obtaining file:///content/GNP
  Preparing metadata (setup.py) ... done
  Running setup.py develop for GNP


In [ ]:
!pwd

/content/GNP


In [ ]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 1024 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

Input options. "n" and "location" may be adjusted according to "problem"
{
  "problem": "1d_laplacian",
  "n": 1024,
  "location": "~/data/SuiteSparse/ssget/mat",
  "rhs": "x_all_ones",
  "restart": 10,
  "max_iters": 1000,
  "timeout": null,
  "rtol": 1e-06,
  "preconditioners": [
    "gmres",
    "gnp",
    "amg"
  ],
  "inner_iters": 10,
  "inner_rtol": 1e-06,
  "training_data": "x_mix",
  "m": 40,
  "num_layers": 8,
  "embed": 16,
  "hidden": 32,
  "drop_rate": 0.0,
  "disable_scale_input": false,
  "precision": "float32",
  "batch_size": 16,
  "grad_accu_steps": 1,
  "epochs": 20,
  "lr": 0.001,
  "weight_decay": 0.0,
  "num_workers": 0,
  "model_file": null,
  "save_model": false,
  "ilu_factors_file": null,
  "save_ilu_factors": false,
  "fill_factor": null,
  "bjac_lu_factors_file": null,
  "save_bjac_lu_factors": false,
  "block_size": 32,
  "out_path": "./dump/",
  "out_file_prefix": null,
  "hide_solver_bar": false,
  "hide_training_bar": false
}
/content/GNP/GNP/problems.py

In [ ]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 4096 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

Input options. "n" and "location" may be adjusted according to "problem"
{
  "problem": "1d_laplacian",
  "n": 4096,
  "location": "~/data/SuiteSparse/ssget/mat",
  "rhs": "x_all_ones",
  "restart": 10,
  "max_iters": 1000,
  "timeout": null,
  "rtol": 1e-06,
  "preconditioners": [
    "gmres",
    "gnp",
    "amg"
  ],
  "inner_iters": 10,
  "inner_rtol": 1e-06,
  "training_data": "x_mix",
  "m": 40,
  "num_layers": 8,
  "embed": 16,
  "hidden": 32,
  "drop_rate": 0.0,
  "disable_scale_input": false,
  "precision": "float32",
  "batch_size": 16,
  "grad_accu_steps": 1,
  "epochs": 20,
  "lr": 0.001,
  "weight_decay": 0.0,
  "num_workers": 0,
  "model_file": null,
  "save_model": false,
  "ilu_factors_file": null,
  "save_ilu_factors": false,
  "fill_factor": null,
  "bjac_lu_factors_file": null,
  "save_bjac_lu_factors": false,
  "block_size": 32,
  "out_path": "./dump/",
  "out_file_prefix": null,
  "hide_solver_bar": false,
  "hide_training_bar": false
}
/content/GNP/GNP/problems.py

In [ ]:
!python examples/all_in_one.py \
--problem 1d_laplacian \
-n 16384 \
--preconditioners gmres gnp amg \
--epochs 20 \
--rtol 1e-6 \
--max_iters 1000

Input options. "n" and "location" may be adjusted according to "problem"
{
  "problem": "1d_laplacian",
  "n": 16384,
  "location": "~/data/SuiteSparse/ssget/mat",
  "rhs": "x_all_ones",
  "restart": 10,
  "max_iters": 1000,
  "timeout": null,
  "rtol": 1e-06,
  "preconditioners": [
    "gmres",
    "gnp",
    "amg"
  ],
  "inner_iters": 10,
  "inner_rtol": 1e-06,
  "training_data": "x_mix",
  "m": 40,
  "num_layers": 8,
  "embed": 16,
  "hidden": 32,
  "drop_rate": 0.0,
  "disable_scale_input": false,
  "precision": "float32",
  "batch_size": 16,
  "grad_accu_steps": 1,
  "epochs": 20,
  "lr": 0.001,
  "weight_decay": 0.0,
  "num_workers": 0,
  "model_file": null,
  "save_model": false,
  "ilu_factors_file": null,
  "save_ilu_factors": false,
  "fill_factor": null,
  "bjac_lu_factors_file": null,
  "save_bjac_lu_factors": false,
  "block_size": 32,
  "out_path": "./dump/",
  "out_file_prefix": null,
  "hide_solver_bar": false,
  "hide_training_bar": false
}
/content/GNP/GNP/problems.p

In [ ]:
import pandas as pd

tabla = pd.DataFrame({
    "Tamaño": [
        1024, 1024, 1024,
        4096, 4096, 4096,
        16384, 16384, 16384
    ],
    "Método": [
        "GMRES",
        "GMRES + AMG",
        "GMRES + GNP",
        "GMRES",
        "GMRES + AMG",
        "GMRES + GNP",
        "GMRES",
        "GMRES + AMG",
        "GMRES + GNP"
    ],
    "Residual Final": [
        5.4369e-04,
        1.3410e-07,
        8.2160e-02,
        5.4369e-04,
        6.5822e-07,
        5.9118e-02,
        5.4369e-04,
        6.6308e-07,
        2.9443e-01
    ],
    "Tiempo (s)": [
        1,
        0.012,
        1.662,
        1,
        0.049,
        6.704,
        6,
        0.192,
        25.131
    ]
})

display(tabla)

,Tamaño,Método,Residual Final,Tiempo (s)
0,1024,GMRES,5.436900e-04,1.000
1,1024,GMRES + AMG,1.341000e-07,0.012
2,1024,GMRES + GNP,8.216000e-02,1.662
3,4096,GMRES,5.436900e-04,1.000
4,4096,GMRES + AMG,6.582200e-07,0.049
5,4096,GMRES + GNP,5.911800e-02,6.704
6,16384,GMRES,5.436900e-04,6.000
7,16384,GMRES + AMG,6.630800e-07,0.192
8,16384,GMRES + GNP,2.944300e-01,25.131


Los resultados obtenidos muestran que el precondicionador AMG mejora significativamente el rendimiento de GMRES para la resolución del problema Laplaciano unidimensional. En todos los tamaños de matriz evaluados, GMRES+AMG alcanza los menores tiempos de ejecución, por otro lado, GMRES sin precondicionamiento mantiene una precisión aceptable, aunque con tiempos mayores para matrices grandes, finalmente, GMRES+GNP presenta tiempos de ejecución considerablemente más altos y residuales significativamente mayores, especialmente para n=16384, donde la calidad de la solución disminuye notablemente. En consecuencia, AMG resulta ser el precondicionador más efectivo para este conjunto de experimentos.

In [ ]:
import time

# ----------------------------
# Entrenamiento
# ----------------------------
inicio_train = time.time()

# código que entrena GNP

fin_train = time.time()

print(f"Tiempo de entrenamiento: {fin_train - inicio_train:.3f} s")

Tiempo de entrenamiento: 0.000 s
